### Structured Output 

##### pydantic

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.8-flash")
model

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="The year the movie was released")
    rating:float = Field(..., description="The rating of the movie on a scale of 1 to 10")
    director:str = Field(..., description="The director of the movie")

structured_model = model.with_structured_output(Movie, include_raw=True)
structured_model.invoke("Can you provide me with the details of the movie Inception?")

In [ ]:

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str = Field(..., description="The name of the actor")
    age:int = Field(..., description="The age of the actor")

class MoviesWithActors(BaseModel):
    title:str = Field(..., description="The title of the movie")
    actors:list[Actor] = Field(..., description="A list of actors in the movie")
    rating:float = Field(..., description="The rating of the movie on a scale of 1 to 10")

updated_structure_model = model.with_structured_output(MoviesWithActors, include_raw=True)
updated_structure_model.invoke("Can you provide me with the details of the movie Inception?")



In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain.chat_models import init_chat_model

class ActorDict(TypedDict):
    name: Annotated[str,"The name of the actor"]
    age: Annotated[int, "The age of the actor"]

class MoviesWithActorsDict(TypedDict):
    title: Annotated[str, "The title of the movie"]
    actors: Annotated[list[ActorDict], "A list of actors in the movie"]
    rating: Annotated[float, "The rating of the movie on a scale of 1 to 10"]

model = init_chat_model("google_genai:gemini-3.8-flash")
updated_structure_model = model.with_structured_output(MoviesWithActorsDict, include_raw=True)
updated_structure_model.invoke("Can you provide me with the details of the movie Inception?")


In [ ]:
from dataclasses import dataclass, field
from langchain.agents import create_agent

@dataclass
class ActorDataClass:
    name:str = field(metadata={"description":"The name of the actor"})
    age:int = field(metadata={"description":"The age of the actor"})


agent = create_agent(
    model="google_genai:gemini-3.8-flash",
    response_format=ActorDataClass
)

message= { "messages": [{"role": "user", "content": "Can you provide me with the details of the actor Leonardo DiCaprio?"}] }
agent.invoke(message)
